In [25]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd

# Предобработка транзакций

In [ ]:
INPUT_PATH = Path(r"выгрузка.xlsx")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

df_raw = pd.read_excel(INPUT_PATH)

print(df_raw.shape)
df_raw.head(3)

(810211, 14)


,Время транзакции,Код клиента,Тип карты,Наименование категории тарифа,Наименование номенклатуры,Объем,Цена стелы,Цена клиента,Тариф клиента,Тип тарифа клиента,Номер АЗС,Регион АЗС,Тип АЗС,Адрес АЗС
0,2025-01-31 16:17:00,CRRRA668R,Поставщик1.пластик-2,Поставщик1-2,Бензин A95,7.99,62.59,63.84,1.02,Проценты,11238,Нижегородская область,Собственная,"Нижний Новгород, Советский район, проспект Гаг..."
1,2025-01-31 13:32:00,CRRRZAXX8,Поставщик1.пластик,Поставщик1-2,Дизельное топливо,45.00,67.19,68.53,1.02,Проценты,11238,Нижегородская область,Собственная,"Нижний Новгород, Советский район, проспект Гаг..."
2,2025-01-31 06:57:00,CRRRA7ZA7,Поставщик1.пластик,Поставщик1-2,Дизельное топливо,10.00,67.19,68.53,1.02,Проценты,11238,Нижегородская область,Собственная,"Нижний Новгород, Советский район, проспект Гаг..."


In [ ]:
df_raw[df_raw.duplicated() == True]

,Время транзакции,Код клиента,Тип карты,Наименование категории тарифа,Наименование номенклатуры,Объем,Цена стелы,Цена клиента,Тариф клиента,Тип тарифа клиента,Номер АЗС,Регион АЗС,Тип АЗС,Адрес АЗС
285,2025-01-04 05:40:00,CRRRZBWAX,Поставщик4.пластик,Поставщик4-105,Дизельное топливо,143.58,68.95,69.64,1.01,Проценты,523,Ленинградская область,Собственная,"Россия, Ленинградская область, Всеволожский р-..."
2413,2025-01-11 15:45:00,CRRRZB8AB,Поставщик2.пластик,Поставщик2-101,Бензин АИ95 G-drive,57.23,61.56,62.18,1.01,Проценты,125,Владимирская область,Собственная,"Владимирская область, Владимир, Суздальский пр..."
2414,2025-01-11 15:45:00,CRRRZB8AB,Поставщик2.пластик,Поставщик2-101,Прод. товары,1.00,139.00,139.00,1.00,Проценты,125,Владимирская область,Собственная,"Владимирская область, Владимир, Суздальский пр..."
2415,2025-01-11 15:45:00,CRRRZB8AB,Поставщик2.пластик,Поставщик2-101,Прод. товары,1.00,179.00,179.00,1.00,Проценты,125,Владимирская область,Собственная,"Владимирская область, Владимир, Суздальский пр..."
4704,2025-01-16 19:45:00,CRRRZZA8B,Поставщик2.пластик,Поставщик2-101,Дизельное топливо,147.06,67.99,70.03,1.03,Проценты,027,Ярославская область,Собственная,"Ярославская область, г. Рыбинск, ул. Труда, д.116"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
809088,2025-06-01 10:00:00,CRRRZZRAW,Поставщик1.виртуальная,Поставщик1-3,Диз. топливо Экто,-450.00,69.30,65.84,0.95,Проценты,АЗС №52038,Нижегородская область,Собственная,"п.Гавриловка, Зеркальный комплекс на Южном объ..."
809089,2025-06-01 10:00:00,CRRRZZRAW,Поставщик1.виртуальная,Поставщик1-3,Диз. топливо Экто,-450.00,69.30,65.84,0.95,Проценты,АЗС №52038,Нижегородская область,Собственная,"п.Гавриловка, Зеркальный комплекс на Южном объ..."
809090,2025-06-01 10:00:00,CRRRZZRAW,Поставщик1.виртуальная,Поставщик1-3,Диз. топливо Экто,-450.00,69.30,65.84,0.95,Проценты,АЗС №52038,Нижегородская область,Собственная,"п.Гавриловка, Зеркальный комплекс на Южном объ..."
809094,2025-06-01 10:00:00,CRRRZXZZX,Поставщик1.виртуальная,Поставщик1-3,Диз. топливо Экто,-100.00,70.46,66.94,0.95,Проценты,02007 АЗС 7,Республика Башкортостан,Собственная,"Верхнеяркеево, п. В.Яркеево, ул. Бакалинская, 6"


In [ ]:
df_raw[df_raw['Объем'] < 0]

,Время транзакции,Код клиента,Тип карты,Наименование категории тарифа,Наименование номенклатуры,Объем,Цена стелы,Цена клиента,Тариф клиента,Тип тарифа клиента,Номер АЗС,Регион АЗС,Тип АЗС,Адрес АЗС
74,2025-01-01 13:58:00,CRRRA9WAW,Поставщик4.пластик,Поставщик4-102,Бензин A92,-11.00,53.89,53.35,0.99,Проценты,552,Владимирская область,Собственная,"Россия, Владимирская область, г. Ковров, ул. Е..."
78,2025-01-01 15:48:00,CRRRZZ8ZW,Поставщик4.пластик,Поставщик4-103,Бензин A95,-6.40,57.85,58.43,1.01,Проценты,615,"Марий ел, республика",Собственная,"Россия, Марий ел, республика, Звениговский мун..."
85,2025-01-01 18:08:00,CRRRA7AWA,Поставщик4.пластик,Поставщик4-104,Бензин A95,-0.52,60.09,60.09,1.00,Проценты,559,Вологодская область,Собственная,"Россия, Вологодская область, г.Череповец, ул.М..."
88,2025-01-01 20:34:00,CRRRZBBX7,Поставщик4.пластик,Поставщик4-104,Бензин A95,-2.15,60.39,60.39,1.00,Проценты,557,Вологодская область,Собственная,"Россия, Вологодская область, Череповецкий р-н,..."
92,2025-01-01 20:54:00,CRRRZXA96,Поставщик4.пластик,Поставщик4-102,Дизельное топливо,-12.85,66.89,64.88,0.97,Проценты,460,Владимирская область,Собственная,"Россия, Владимирская область, Петушинский р-н,..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
810099,2025-06-27 14:55:00,CRRRZZ7B6,Поставщик3.пластик,Поставщик3-101,Бензин A95,-3.70,60.90,60.29,0.99,Проценты,АЗС РОСНЕФТЬ 041,Рязанская область,Собственная,"Россия, Рязанская область, г. Касимов, ул. Заг..."
810102,2025-06-27 08:06:00,CRRRZX9RX,Поставщик3.пластик,Поставщик3-101,Дизельное топливо,-19.24,70.65,70.65,1.00,Проценты,АЗС РОСНЕФТЬ MJ025,Москва,Собственная,"Россия, Москва, г. Москва, ул. Бутырская, 10 (..."
810121,2025-06-27 10:51:00,CRRRZZ7B6,Поставщик3.пластик,Поставщик3-101,Бензин A95,-0.44,63.99,63.35,0.99,Проценты,АЗС РОСНЕФТЬ 134,Ростовская область,Собственная,"Россия, Ростовская область, г. Морозовск, ул. ..."
810175,2025-06-30 09:19:00,CRRRZZ7B6,Поставщик3.пластик,Поставщик3-101,Бензин A95,-15.00,63.99,63.35,0.99,Проценты,АЗС РОСНЕФТЬ 047,Ростовская область,Собственная,"Россия, Ростовская область, ст-ца Кагальницкая..."


In [ ]:
df_raw['Наименование номенклатуры'].unique()

array(['Бензин A95', 'Дизельное топливо', 'Бензин АИ95 Экто',
       'Бензин АИ92 Экто', 'Диз. топливо Экто', 'Бензин A92',
       'Диз. топливо Танеко', 'Газ Пропан-бутан', 'Бензин A98',
       'Бензин АИ95 G-drive', 'Бензин АИ100 G-drive', 'Прод. товары',
       'Mасло', 'Автохимия', 'Сопутствующие товары', 'Бензин АИ100 Экто',
       'Автотовары', 'AdBlue', 'Автомойка', 'Шиномонтаж', 'КПГ',
       'Бензин АИ98 G-drive'], dtype=object)

In [ ]:
def normalize_fuel_name(raw_name: str) -> str:
    if not isinstance(raw_name, str):
        return 'UNKNOWN'
    n = raw_name.upper().replace('AI', 'АИ').replace('A', 'А')
    n = re.sub(r'\\s+', ' ', n).strip()
    if 'ДИЗ' in n:
        return 'ДТ'
    if '100' in n:
        return 'АИ-100'
    if '98' in n:
        return 'АИ-98'
    if '95' in n:
        return 'АИ-95'
    if '92' in n:
        return 'АИ-92'
    if 'ГАЗ' in n or 'LPG' in n:
        return 'ГАЗ'
    return n

In [ ]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 810211 entries, 0 to 810210
Data columns (total 14 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   Время транзакции               810211 non-null  datetime64[ns]
 1   Код клиента                    810211 non-null  object        
 2   Тип карты                      810211 non-null  object        
 3   Наименование категории тарифа  810211 non-null  object        
 4   Наименование номенклатуры      810211 non-null  object        
 5   Объем                          810211 non-null  float64       
 6   Цена стелы                     810211 non-null  float64       
 7   Цена клиента                   810211 non-null  float64       
 8   Тариф клиента                  810211 non-null  float64       
 9   Тип тарифа клиента             810211 non-null  object        
 10  Номер АЗС                      810004 non-null  object        
 11  

In [ ]:
df_raw.isna().any()

,0
Время транзакции,False
Код клиента,False
Тип карты,False
Наименование категории тарифа,False
Наименование номенклатуры,False
Объем,False
Цена стелы,False
Цена клиента,False
Тариф клиента,False
Тип тарифа клиента,False


In [ ]:
#Удаление дубликатов
df_tx = df_raw.drop_duplicates()

#Стандартизация наименований топлива
fuel_col = "Наименование номенклатуры"
df_tx["Номенклатура"] = df_tx[fuel_col].apply(normalize_fuel_name)


#Все object-колонки приводим к строковому типу
object_cols = df_tx.select_dtypes(include=["object"]).columns.tolist()
for col in object_cols:
    df_tx[col] = df_tx[col].astype("string")

#Очистка от аномалий-транзакции с объемом меньше 0
df_tx = df_tx[df_tx["Объем"] >= 0]

/tmp/ipykernel_11875/2587986159.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_tx["Номенклатура"] = df_tx[fuel_col].apply(normalize_fuel_name)
/tmp/ipykernel_11875/2587986159.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_tx[col] = df_tx[col].astype("string")


In [ ]:
df_tx

,Время транзакции,Код клиента,Тип карты,Наименование категории тарифа,Наименование номенклатуры,Объем,Цена стелы,Цена клиента,Тариф клиента,Тип тарифа клиента,Номер АЗС,Регион АЗС,Тип АЗС,Адрес АЗС,Номенклатура
0,2025-01-31 16:17:00,CRRRA668R,Поставщик1.пластик-2,Поставщик1-2,Бензин A95,7.99,62.59,63.84,1.020,Проценты,11238,Нижегородская область,Собственная,"Нижний Новгород, Советский район, проспект Гаг...",АИ-95
1,2025-01-31 13:32:00,CRRRZAXX8,Поставщик1.пластик,Поставщик1-2,Дизельное топливо,45.00,67.19,68.53,1.020,Проценты,11238,Нижегородская область,Собственная,"Нижний Новгород, Советский район, проспект Гаг...",ДТ
2,2025-01-31 06:57:00,CRRRA7ZA7,Поставщик1.пластик,Поставщик1-2,Дизельное топливо,10.00,67.19,68.53,1.020,Проценты,11238,Нижегородская область,Собственная,"Нижний Новгород, Советский район, проспект Гаг...",ДТ
3,2025-01-31 23:33:00,CRRRA68RA,Поставщик1.пластик,Поставщик1-0,Бензин АИ95 Экто,60.00,61.99,63.23,1.020,Проценты,АЗС №33039,Владимирская область,Собственная,"д.Сенино, 244 км трассы М7, слева при движении...",АИ-95
4,2025-01-31 18:20:00,CRRRA7Z67,Поставщик1.пластик,Поставщик1-0,Бензин АИ95 Экто,44.53,61.99,64.16,1.035,Проценты,АЗС №33039,Владимирская область,Собственная,"д.Сенино, 244 км трассы М7, слева при движении...",АИ-95
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
810206,2025-06-30 17:43:00,CRRRZZ7B6,Поставщик3.пластик,Поставщик3-101,Бензин A95,17.63,59.65,59.05,0.990,Проценты,АЗС РОСНЕФТЬ 057,Республика Хакасия,Собственная,"Россия, Республика Хакасия, с. Шира, ул. Курор...",АИ-95
810207,2025-06-30 18:51:00,CRRRZX9RX,Поставщик3.пластик,Поставщик3-101,Дизельное топливо,100.00,70.45,70.45,1.000,Проценты,АЗС РОСНЕФТЬ MJ056,Московская область,Собственная,"Россия, Московская область, г. Мытищи, ул. Тру...",ДТ
810208,2025-06-30 20:06:00,CRRRZZ7B6,Поставщик3.пластик,Поставщик3-102,Бензин A95,20.00,57.05,56.48,0.990,Проценты,АЗС РОСНЕФТЬ 027,Алтайский край,Собственная,"Россия, Алтайский край, г. Камень-на-Оби, ул. ...",АИ-95
810209,2025-06-26 22:46:00,CRRRZ6RW6,Поставщик1.пластик,Поставщик1-0,Прод. товары,1.00,159.00,131.97,0.830,Проценты,АЗС №01263,Республика Адыгея,Собственная,"Краснодар, Автотрасса А-146 1-3 км. справа, -,...",ПРОД. ТОВАРЫ


In [ ]:
missing_mask = df_tx.isna().any(axis=1)
missing_mask

,0
0,False
1,False
2,False
3,False
4,False
...,...
810206,False
810207,False
810208,False
810209,False


In [ ]:
display(df_tx.isna().sum().sort_values(ascending=False))

,0
Номер АЗС,194
Регион АЗС,127
Адрес АЗС,1
Наименование категории тарифа,0
Время транзакции,0
Код клиента,0
Тип карты,0
Цена стелы,0
Объем,0
Наименование номенклатуры,0


In [ ]:
df_tx = df_tx.dropna().copy()

In [ ]:
df_tx

,Время транзакции,Код клиента,Тип карты,Наименование категории тарифа,Наименование номенклатуры,Объем,Цена стелы,Цена клиента,Тариф клиента,Тип тарифа клиента,Номер АЗС,Регион АЗС,Тип АЗС,Адрес АЗС,Номенклатура
0,2025-01-31 16:17:00,CRRRA668R,Поставщик1.пластик-2,Поставщик1-2,Бензин A95,7.99,62.59,63.84,1.020,Проценты,11238,Нижегородская область,Собственная,"Нижний Новгород, Советский район, проспект Гаг...",АИ-95
1,2025-01-31 13:32:00,CRRRZAXX8,Поставщик1.пластик,Поставщик1-2,Дизельное топливо,45.00,67.19,68.53,1.020,Проценты,11238,Нижегородская область,Собственная,"Нижний Новгород, Советский район, проспект Гаг...",ДТ
2,2025-01-31 06:57:00,CRRRA7ZA7,Поставщик1.пластик,Поставщик1-2,Дизельное топливо,10.00,67.19,68.53,1.020,Проценты,11238,Нижегородская область,Собственная,"Нижний Новгород, Советский район, проспект Гаг...",ДТ
3,2025-01-31 23:33:00,CRRRA68RA,Поставщик1.пластик,Поставщик1-0,Бензин АИ95 Экто,60.00,61.99,63.23,1.020,Проценты,АЗС №33039,Владимирская область,Собственная,"д.Сенино, 244 км трассы М7, слева при движении...",АИ-95
4,2025-01-31 18:20:00,CRRRA7Z67,Поставщик1.пластик,Поставщик1-0,Бензин АИ95 Экто,44.53,61.99,64.16,1.035,Проценты,АЗС №33039,Владимирская область,Собственная,"д.Сенино, 244 км трассы М7, слева при движении...",АИ-95
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
810206,2025-06-30 17:43:00,CRRRZZ7B6,Поставщик3.пластик,Поставщик3-101,Бензин A95,17.63,59.65,59.05,0.990,Проценты,АЗС РОСНЕФТЬ 057,Республика Хакасия,Собственная,"Россия, Республика Хакасия, с. Шира, ул. Курор...",АИ-95
810207,2025-06-30 18:51:00,CRRRZX9RX,Поставщик3.пластик,Поставщик3-101,Дизельное топливо,100.00,70.45,70.45,1.000,Проценты,АЗС РОСНЕФТЬ MJ056,Московская область,Собственная,"Россия, Московская область, г. Мытищи, ул. Тру...",ДТ
810208,2025-06-30 20:06:00,CRRRZZ7B6,Поставщик3.пластик,Поставщик3-102,Бензин A95,20.00,57.05,56.48,0.990,Проценты,АЗС РОСНЕФТЬ 027,Алтайский край,Собственная,"Россия, Алтайский край, г. Камень-на-Оби, ул. ...",АИ-95
810209,2025-06-26 22:46:00,CRRRZ6RW6,Поставщик1.пластик,Поставщик1-0,Прод. товары,1.00,159.00,131.97,0.830,Проценты,АЗС №01263,Республика Адыгея,Собственная,"Краснодар, Автотрасса А-146 1-3 км. справа, -,...",ПРОД. ТОВАРЫ


In [ ]:
df_tx.to_excel('df_tx.xlsx', index=False)



# Клиенты. Предобработка

In [ ]:
#Таблица клиентов

WORKBOOK_PATH = Path("выгрузка.xlsx")
CLIENTS_SHEET = "Клиенты"
df_clients = pd.read_excel(WORKBOOK_PATH, sheet_name=CLIENTS_SHEET)


df_clients[df_clients['Дата окончания договора'] != "-"]

,Код клиента,Дата начала договора,Дата окончания договора,Статус договора,Тип клиента,Офис обслуживания,Вид деятельности клиента,Отгрузки c 01.01.2025 до 01.07.2025,"Заявленный объем в месяц, л"
11,CRRRABXZB,2026-01-01,2026-04-02 00:00:00,Активен,Бюджетник,ОП Нижний Новгород,NaN,179570.39,0
12,CRRRAB88A,2025-12-01,2026-12-31 00:00:00,Активен,Бюджетник,ОП Владимир,NaN,464.45,0
13,CRRRAWB6X,2022-01-01,2026-12-31 00:00:00,Активен,Бюджетник,ОП Нижний Новгород,NaN,347.39,0
16,CRRRAWWZX,2026-04-01,2026-06-30 00:00:00,Активен,Бюджетник,ОП Нижний Новгород,NaN,17016.94,0
17,CRRRAWWZ6,2025-10-01,2025-12-31 00:00:00,Активен,Бюджетник,ОП Нижний Новгород,NaN,9437.24,0
...,...,...,...,...,...,...,...,...,...
4388,CRRRBAZ9X,2016-11-23,2026-03-25 00:00:00,Заблокирован,Коммерческий,ОП Череповец,NaN,174.05,0
4401,CRRRBAB9X,2017-03-22,2025-10-23 00:00:00,Заблокирован,Коммерческий,ОП Волжск,NaN,42.26,0
4403,CRRRBAWRB,2017-03-14,2026-03-19 00:00:00,Заблокирован,Коммерческий,ОП Чебоксары,NaN,162.15,0
4416,CRRRBAWWZ,2017-04-14,2025-10-17 00:00:00,Заблокирован,Коммерческий,ОП Владимир,Розничная торговля,2939.67,0


In [ ]:
df_clients.duplicated().any()

np.False_

In [ ]:
df_clients["Вид деятельности клиента"] = df_clients["Вид деятельности клиента"].fillna("Не указан")

df_clients["Заявленный объем в месяц, л"] = pd.to_numeric(
    df_clients["Заявленный объем в месяц, л"], errors="coerce"
).fillna(0)
df_clients["Отгрузки c 01.01.2025 до 01.07.2025"] = pd.to_numeric(
    df_clients["Отгрузки c 01.01.2025 до 01.07.2025"], errors="coerce"
).fillna(0)

#Проверка корректности дат договора
df_clients["Дата начала договора"] = pd.to_datetime(df_clients["Дата начала договора"])
df_clients["Дата окончания договора"] = pd.to_datetime(df_clients["Дата окончания договора"])

invalid_order = df_clients[
    df_clients["Дата окончания договора"] < df_clients["Дата начала договора"]
].copy()

#Договор должен действовать в период отгрузок
shipment_start, shipment_end = None, None
if "df_tx" in globals() and "Время транзакции" in df_tx.columns:
    tx_dates = pd.to_datetime(df_tx["Время транзакции"])
    shipment_start, shipment_end = tx_dates.min(), tx_dates.max()
elif "raw_tx" in globals() and "Время транзакции" in raw_tx.columns:
    tx_dates = pd.to_datetime(raw_tx["Время транзакции"])
    shipment_start, shipment_end = tx_dates.min(), tx_dates.max()
elif "df_raw" in globals() and "Время транзакции" in df_raw.columns:
    tx_dates = pd.to_datetime(df_raw["Время транзакции"])
    shipment_start, shipment_end = tx_dates.min(), tx_dates.max()

invalid_active_period = pd.DataFrame()
if shipment_start is not None and shipment_end is not None:
    # Нет пересечения периода договора с периодом отгрузок
    invalid_active_period = df_clients[
        (df_clients["Дата окончания договора"] < shipment_start)
        | (df_clients["Дата начала договора"] > shipment_end)
    ].copy()

#object в string
obj_cols = df_clients.select_dtypes(include=["object"]).columns.tolist()
for col in obj_cols:
    df_clients[col] = df_clients[col].astype("string")

,Код клиента,Дата начала договора,Дата окончания договора,Статус договора,Тип клиента,Офис обслуживания,Вид деятельности клиента,Отгрузки c 01.01.2025 до 01.07.2025,"Заявленный объем в месяц, л"
11,CRRRABXZB,2026-01-01,2026-04-02,Активен,Бюджетник,ОП Нижний Новгород,Не указан,179570.39,0
12,CRRRAB88A,2025-12-01,2026-12-31,Активен,Бюджетник,ОП Владимир,Не указан,464.45,0
16,CRRRAWWZX,2026-04-01,2026-06-30,Активен,Бюджетник,ОП Нижний Новгород,Не указан,17016.94,0
17,CRRRAWWZ6,2025-10-01,2025-12-31,Активен,Бюджетник,ОП Нижний Новгород,Не указан,9437.24,0
20,CRRRAW6WX,2026-01-01,2026-12-31,Активен,Бюджетник,ОП Нижний Новгород,Не указан,785.00,0
29,CRRRAW977,2026-03-27,2026-12-31,Активен,Бюджетник,ОП Нижний Новгород,Не указан,14750.11,0
38,CRRRAXABX,2026-01-01,2026-07-01,Активен,Бюджетник,ОП Нижний Новгород,Не указан,4559.92,0
44,CRRRAXB9X,2026-01-01,2026-12-31,Активен,Бюджетник,ОП Нижний Новгород,Не указан,5125.01,0
72,CRRRAX7X8,2026-01-01,2026-07-31,Активен,Бюджетник,ОП Нижний Новгород,Не указан,16226.61,0
75,CRRRAX76Z,2025-12-16,NaT,Активен,Коммерческий,ОП Волжск,Распределение газообразного топлива по газорас...,16491.05,0


,0
Код клиента,string[python]
Дата начала договора,datetime64[ns]
Дата окончания договора,datetime64[ns]
Статус договора,string[python]
Тип клиента,string[python]
Офис обслуживания,string[python]
Вид деятельности клиента,string[python]
Отгрузки c 01.01.2025 до 01.07.2025,float64
"Заявленный объем в месяц, л",int64


In [ ]:
len(df_clients[df_clients['Тип клиента'] == "Бюджетник"])

336

In [ ]:
#Удаляем клиентов-бюджетников
df_clients = df_clients[df_clients["Тип клиента"] != "Бюджетик"].copy()

In [ ]:
df_clients.to_excel('df_clients.xlsx', index=False)

# Feature Engineering

In [13]:
import pandas as pd
import numpy as np

In [15]:
clients = pd.read_excel('df_clients.xlsx')
clients

,Код клиента,Дата начала договора,Дата окончания договора,Статус договора,Тип клиента,Офис обслуживания,Вид деятельности клиента,Отгрузки c 01.01.2025 до 01.07.2025,"Заявленный объем в месяц, л"
0,CRRRAAW96,2019-01-01,NaT,Активен,Коммерческий,ОП Нижний Новгород,Деятельность автомобильного грузового транспор...,95959.51,0
1,CRRRAA8XR,2019-01-01,NaT,Активен,Коммерческий,ОП Нижний Новгород,Аренда спецтехники,6192.14,0
2,CRRRAA87A,2019-01-01,NaT,Активен,Коммерческий,ОП Нижний Новгород,"Производство варенья, топингов, фрукры, консер...",46244.00,0
3,CRRRAZAZ9,2019-01-01,NaT,Активен,Коммерческий,ОП Нижний Новгород,Торгуют автозапчастями,605.89,0
4,CRRRAZAWR,2019-01-01,NaT,Активен,Коммерческий,ОП Нижний Новгород,Клиент занимается сдачей недвижимости в аренду...,817.78,0
...,...,...,...,...,...,...,...,...,...
4423,CRRRBAW8R,2017-05-10,NaT,Активен,Коммерческий,ОП Вологда,Не указан,1759.84,0
4424,CRRRBAW8A,2017-05-11,NaT,Активен,Коммерческий,ОП Вологда,Не указан,1987.73,0
4425,CRRRBAW9A,2017-05-16,NaT,Активен,Коммерческий,ОП Владимир,Производство мет. Конструкций,15480.00,0
4426,CRRRBAW9W,2017-05-17,NaT,Активен,Коммерческий,ОП Чебоксары,Не указан,4153.05,0


In [16]:
tx = pd.read_excel('df_tx.xlsx')
tx

,Время транзакции,Код клиента,Тип карты,Наименование категории тарифа,Наименование номенклатуры,Объем,Цена стелы,Цена клиента,Тариф клиента,Тип тарифа клиента,Номер АЗС,Регион АЗС,Тип АЗС,Адрес АЗС,Номенклатура
0,2025-01-31 16:17:00,CRRRA668R,Поставщик1.пластик-2,Поставщик1-2,Бензин A95,7.99,62.59,63.84,1.020,Проценты,11238,Нижегородская область,Собственная,"Нижний Новгород, Советский район, проспект Гаг...",АИ-95
1,2025-01-31 13:32:00,CRRRZAXX8,Поставщик1.пластик,Поставщик1-2,Дизельное топливо,45.00,67.19,68.53,1.020,Проценты,11238,Нижегородская область,Собственная,"Нижний Новгород, Советский район, проспект Гаг...",ДТ
2,2025-01-31 06:57:00,CRRRA7ZA7,Поставщик1.пластик,Поставщик1-2,Дизельное топливо,10.00,67.19,68.53,1.020,Проценты,11238,Нижегородская область,Собственная,"Нижний Новгород, Советский район, проспект Гаг...",ДТ
3,2025-01-31 23:33:00,CRRRA68RA,Поставщик1.пластик,Поставщик1-0,Бензин АИ95 Экто,60.00,61.99,63.23,1.020,Проценты,АЗС №33039,Владимирская область,Собственная,"д.Сенино, 244 км трассы М7, слева при движении...",АИ-95
4,2025-01-31 18:20:00,CRRRA7Z67,Поставщик1.пластик,Поставщик1-0,Бензин АИ95 Экто,44.53,61.99,64.16,1.035,Проценты,АЗС №33039,Владимирская область,Собственная,"д.Сенино, 244 км трассы М7, слева при движении...",АИ-95
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
788899,2025-06-30 17:43:00,CRRRZZ7B6,Поставщик3.пластик,Поставщик3-101,Бензин A95,17.63,59.65,59.05,0.990,Проценты,АЗС РОСНЕФТЬ 057,Республика Хакасия,Собственная,"Россия, Республика Хакасия, с. Шира, ул. Курор...",АИ-95
788900,2025-06-30 18:51:00,CRRRZX9RX,Поставщик3.пластик,Поставщик3-101,Дизельное топливо,100.00,70.45,70.45,1.000,Проценты,АЗС РОСНЕФТЬ MJ056,Московская область,Собственная,"Россия, Московская область, г. Мытищи, ул. Тру...",ДТ
788901,2025-06-30 20:06:00,CRRRZZ7B6,Поставщик3.пластик,Поставщик3-102,Бензин A95,20.00,57.05,56.48,0.990,Проценты,АЗС РОСНЕФТЬ 027,Алтайский край,Собственная,"Россия, Алтайский край, г. Камень-на-Оби, ул. ...",АИ-95
788902,2025-06-26 22:46:00,CRRRZ6RW6,Поставщик1.пластик,Поставщик1-0,Прод. товары,1.00,159.00,131.97,0.830,Проценты,АЗС №01263,Республика Адыгея,Собственная,"Краснодар, Автотрасса А-146 1-3 км. справа, -,...",ПРОД. ТОВАРЫ


In [17]:
tx[['Поставщик', 'Канал']] = tx['Тип карты'].str.split('.', expand=True)

In [18]:
tx

,Время транзакции,Код клиента,Тип карты,Наименование категории тарифа,Наименование номенклатуры,Объем,Цена стелы,Цена клиента,Тариф клиента,Тип тарифа клиента,Номер АЗС,Регион АЗС,Тип АЗС,Адрес АЗС,Номенклатура,Поставщик,Канал
0,2025-01-31 16:17:00,CRRRA668R,Поставщик1.пластик-2,Поставщик1-2,Бензин A95,7.99,62.59,63.84,1.020,Проценты,11238,Нижегородская область,Собственная,"Нижний Новгород, Советский район, проспект Гаг...",АИ-95,Поставщик1,пластик-2
1,2025-01-31 13:32:00,CRRRZAXX8,Поставщик1.пластик,Поставщик1-2,Дизельное топливо,45.00,67.19,68.53,1.020,Проценты,11238,Нижегородская область,Собственная,"Нижний Новгород, Советский район, проспект Гаг...",ДТ,Поставщик1,пластик
2,2025-01-31 06:57:00,CRRRA7ZA7,Поставщик1.пластик,Поставщик1-2,Дизельное топливо,10.00,67.19,68.53,1.020,Проценты,11238,Нижегородская область,Собственная,"Нижний Новгород, Советский район, проспект Гаг...",ДТ,Поставщик1,пластик
3,2025-01-31 23:33:00,CRRRA68RA,Поставщик1.пластик,Поставщик1-0,Бензин АИ95 Экто,60.00,61.99,63.23,1.020,Проценты,АЗС №33039,Владимирская область,Собственная,"д.Сенино, 244 км трассы М7, слева при движении...",АИ-95,Поставщик1,пластик
4,2025-01-31 18:20:00,CRRRA7Z67,Поставщик1.пластик,Поставщик1-0,Бензин АИ95 Экто,44.53,61.99,64.16,1.035,Проценты,АЗС №33039,Владимирская область,Собственная,"д.Сенино, 244 км трассы М7, слева при движении...",АИ-95,Поставщик1,пластик
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
788899,2025-06-30 17:43:00,CRRRZZ7B6,Поставщик3.пластик,Поставщик3-101,Бензин A95,17.63,59.65,59.05,0.990,Проценты,АЗС РОСНЕФТЬ 057,Республика Хакасия,Собственная,"Россия, Республика Хакасия, с. Шира, ул. Курор...",АИ-95,Поставщик3,пластик
788900,2025-06-30 18:51:00,CRRRZX9RX,Поставщик3.пластик,Поставщик3-101,Дизельное топливо,100.00,70.45,70.45,1.000,Проценты,АЗС РОСНЕФТЬ MJ056,Московская область,Собственная,"Россия, Московская область, г. Мытищи, ул. Тру...",ДТ,Поставщик3,пластик
788901,2025-06-30 20:06:00,CRRRZZ7B6,Поставщик3.пластик,Поставщик3-102,Бензин A95,20.00,57.05,56.48,0.990,Проценты,АЗС РОСНЕФТЬ 027,Алтайский край,Собственная,"Россия, Алтайский край, г. Камень-на-Оби, ул. ...",АИ-95,Поставщик3,пластик
788902,2025-06-26 22:46:00,CRRRZ6RW6,Поставщик1.пластик,Поставщик1-0,Прод. товары,1.00,159.00,131.97,0.830,Проценты,АЗС №01263,Республика Адыгея,Собственная,"Краснодар, Автотрасса А-146 1-3 км. справа, -,...",ПРОД. ТОВАРЫ,Поставщик1,пластик


In [22]:
tx['Время транзакции'] = pd.to_datetime(tx['Время транзакции'])

tx['год']        = tx['Время транзакции'].dt.year
tx['месяц']      = tx['Время транзакции'].dt.month
tx['неделя']     = tx['Время транзакции'].dt.isocalendar().week.astype(int)
tx['час']        = tx['Время транзакции'].dt.hour
tx['день_недели'] = tx['Время транзакции'].dt.dayofweek
tx["выходной"] = (tx["день_недели"] >= 5).astype(int)
tx["ночь"] = ((tx["час"] < 6) | (tx["час"] >= 22)).astype(int)

In [23]:
tx

,Время транзакции,Код клиента,Тип карты,Наименование категории тарифа,Наименование номенклатуры,Объем,Цена стелы,Цена клиента,Тариф клиента,Тип тарифа клиента,...,Номенклатура,Поставщик,Канал,год,месяц,неделя,час,день_недели,выходной,ночь
0,2025-01-31 16:17:00,CRRRA668R,Поставщик1.пластик-2,Поставщик1-2,Бензин A95,7.99,62.59,63.84,1.020,Проценты,...,АИ-95,Поставщик1,пластик-2,2025,1,5,16,4,0,0
1,2025-01-31 13:32:00,CRRRZAXX8,Поставщик1.пластик,Поставщик1-2,Дизельное топливо,45.00,67.19,68.53,1.020,Проценты,...,ДТ,Поставщик1,пластик,2025,1,5,13,4,0,0
2,2025-01-31 06:57:00,CRRRA7ZA7,Поставщик1.пластик,Поставщик1-2,Дизельное топливо,10.00,67.19,68.53,1.020,Проценты,...,ДТ,Поставщик1,пластик,2025,1,5,6,4,0,0
3,2025-01-31 23:33:00,CRRRA68RA,Поставщик1.пластик,Поставщик1-0,Бензин АИ95 Экто,60.00,61.99,63.23,1.020,Проценты,...,АИ-95,Поставщик1,пластик,2025,1,5,23,4,0,1
4,2025-01-31 18:20:00,CRRRA7Z67,Поставщик1.пластик,Поставщик1-0,Бензин АИ95 Экто,44.53,61.99,64.16,1.035,Проценты,...,АИ-95,Поставщик1,пластик,2025,1,5,18,4,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
788899,2025-06-30 17:43:00,CRRRZZ7B6,Поставщик3.пластик,Поставщик3-101,Бензин A95,17.63,59.65,59.05,0.990,Проценты,...,АИ-95,Поставщик3,пластик,2025,6,27,17,0,0,0
788900,2025-06-30 18:51:00,CRRRZX9RX,Поставщик3.пластик,Поставщик3-101,Дизельное топливо,100.00,70.45,70.45,1.000,Проценты,...,ДТ,Поставщик3,пластик,2025,6,27,18,0,0,0
788901,2025-06-30 20:06:00,CRRRZZ7B6,Поставщик3.пластик,Поставщик3-102,Бензин A95,20.00,57.05,56.48,0.990,Проценты,...,АИ-95,Поставщик3,пластик,2025,6,27,20,0,0,0
788902,2025-06-26 22:46:00,CRRRZ6RW6,Поставщик1.пластик,Поставщик1-0,Прод. товары,1.00,159.00,131.97,0.830,Проценты,...,ПРОД. ТОВАРЫ,Поставщик1,пластик,2025,6,26,22,3,0,1


In [26]:
#Тарифный план: 'Поставщик1-0' -> '0', 'Поставщик2-101' -> '101'
tx["Тарифный_план"] = (tx["Наименование категории тарифа"].astype(str).apply(lambda s: re.split(r"-", s, maxsplit=1)[1] if "-" in s else "0"))

In [33]:
# 6. Тип АЗС в категориальную форму
tx["Тип АЗС"] =(tx["Тип АЗС"].astype(str).str.lower() == "собственная").astype(int)

In [35]:
tx

,Время транзакции,Код клиента,Тип карты,Наименование категории тарифа,Наименование номенклатуры,Объем,Цена стелы,Цена клиента,Тариф клиента,Тип тарифа клиента,...,Поставщик,Канал,год,месяц,неделя,час,день_недели,выходной,ночь,Тарифный_план
0,2025-01-31 16:17:00,CRRRA668R,Поставщик1.пластик-2,Поставщик1-2,Бензин A95,7.99,62.59,63.84,1.020,Проценты,...,Поставщик1,пластик-2,2025,1,5,16,4,0,0,2
1,2025-01-31 13:32:00,CRRRZAXX8,Поставщик1.пластик,Поставщик1-2,Дизельное топливо,45.00,67.19,68.53,1.020,Проценты,...,Поставщик1,пластик,2025,1,5,13,4,0,0,2
2,2025-01-31 06:57:00,CRRRA7ZA7,Поставщик1.пластик,Поставщик1-2,Дизельное топливо,10.00,67.19,68.53,1.020,Проценты,...,Поставщик1,пластик,2025,1,5,6,4,0,0,2
3,2025-01-31 23:33:00,CRRRA68RA,Поставщик1.пластик,Поставщик1-0,Бензин АИ95 Экто,60.00,61.99,63.23,1.020,Проценты,...,Поставщик1,пластик,2025,1,5,23,4,0,1,0
4,2025-01-31 18:20:00,CRRRA7Z67,Поставщик1.пластик,Поставщик1-0,Бензин АИ95 Экто,44.53,61.99,64.16,1.035,Проценты,...,Поставщик1,пластик,2025,1,5,18,4,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
788899,2025-06-30 17:43:00,CRRRZZ7B6,Поставщик3.пластик,Поставщик3-101,Бензин A95,17.63,59.65,59.05,0.990,Проценты,...,Поставщик3,пластик,2025,6,27,17,0,0,0,101
788900,2025-06-30 18:51:00,CRRRZX9RX,Поставщик3.пластик,Поставщик3-101,Дизельное топливо,100.00,70.45,70.45,1.000,Проценты,...,Поставщик3,пластик,2025,6,27,18,0,0,0,101
788901,2025-06-30 20:06:00,CRRRZZ7B6,Поставщик3.пластик,Поставщик3-102,Бензин A95,20.00,57.05,56.48,0.990,Проценты,...,Поставщик3,пластик,2025,6,27,20,0,0,0,102
788902,2025-06-26 22:46:00,CRRRZ6RW6,Поставщик1.пластик,Поставщик1-0,Прод. товары,1.00,159.00,131.97,0.830,Проценты,...,Поставщик1,пластик,2025,6,26,22,3,0,1,0


In [56]:
tx["amount"] = tx["Объем"] * tx["Цена клиента"]
tx["amount_market"]= tx["Объем"] * tx["Цена стелы"]
tx["saving"]= tx["amount_market"] - tx["amount"]
tx["price_ratio"]=tx["Цена клиента"] / tx["Цена стелы"]


In [38]:
def clean_clients(clients: pd.DataFrame) -> pd.DataFrame:
    """Очистка clients + категориальные поля"""
    df = clients.copy()
    df["договор_бессрочный"] = df["Дата окончания договора"].isna().astype(int)
    df["заявленный_объем_указан"] = (df["Заявленный объем в месяц, л"] > 0).astype(int)
    return df

In [41]:
def _row_entropy(p: np.ndarray) -> np.ndarray:
    """Шенноновская энтропия по строкам матрицы вероятностей."""
    with np.errstate(divide="ignore", invalid="ignore"):
        return -np.where(p > 0, p * np.log(p), 0).sum(axis=1)

In [42]:
def _price_sensitivity(group: pd.DataFrame) -> float:
    """
    Корреляция Цена клиента ↔ Объём по транзакциям клиента. -1 = очень чувствителен к цене, 0 = безразличен.
    """
    if len(group) < 5:
        return 0.0
    x = group["Цена клиента"].values
    y = group["Объем"].values
    if x.std() == 0 or y.std() == 0:
        return 0.0
    return float(np.corrcoef(x, y)[0, 1])

In [59]:
#Создаем профиль клиента

def build_client_profile(tx: pd.DataFrame,
                         clients: pd.DataFrame) -> pd.DataFrame:
    """ 1 строка на клиента. Блоки признаков:
      1. Объёмные:сколько и как часто.
      2. Финансовые: сколько платит, сколько экономит.
      3. Топливные: что покупает (доли по типам, энтропия).
      4. География: где покупает, концентрация (HHI).
      5. Время: когда (ночь/день, будни/выходные).
      6. Поведение: стабильность, лояльность поставщику.
      7. Из справочника: тип, офис, возраст договора, выполнение объемов.
      8. Сегмент: XL/L/M/S по объёму.
    """
    g = tx.groupby("Код клиента")

    #Объёмные
    profile = g.agg(
        tx_count=("Объем", "size"),
        total_volume=("Объем", "sum"),
        avg_volume_per_tx=("Объем", "mean"),
        median_volume=("Объем", "median"),
        std_volume=("Объем", "std"),
        first_tx=("Время транзакции", "min"),
        last_tx=("Время транзакции", "max"))
    profile["std_volume"] = profile["std_volume"].fillna(0)
    profile["activity_days"] = (profile["last_tx"] - profile["first_tx"]).dt.days.clip(lower=1)
    profile["tx_per_day"] = profile["tx_count"] / profile["activity_days"]

    #Финансовые
    fin = g.agg(
        total_spend=("amount", "sum"),
        total_market_spend=("amount_market", "sum"),
        total_saving=("saving", "sum"),
        avg_ticket=("amount", "mean"),
        avg_price_ratio=("price_ratio", "mean"),
    )
    fin["saving_pct"] = (fin["total_saving"] / fin["total_market_spend"]).fillna(0)
    profile = profile.join(fin)

    #Таблица по типа топлива
    fuel_mix = (tx.groupby(["Код клиента", "Номенклатура"])["Объем"].sum().unstack(fill_value=0))
    fuel_mix = fuel_mix.div(fuel_mix.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
    fuel_mix.columns = [f"share_{c}" for c in fuel_mix.columns]
    profile = profile.join(fuel_mix)

    profile["fuel_entropy"] = _row_entropy(fuel_mix.values)
    profile["fuel_diversity"] = g["Номенклатура"].nunique()

    #География
    geo = g.agg(
        unique_regions=("Регион АЗС", "nunique"),
        unique_stations=("Номер АЗС", "nunique"),
        own_station_share=("Тип АЗС", "mean"),
    )
    profile = profile.join(geo)

    #HHI по регионам: 1 = всё в одном регионе, ~0 = равномерно
    region_share = (
        tx.groupby(["Код клиента", "Регион АЗС"])["Объем"].sum().groupby(level=0).apply(lambda s: ((s / s.sum()) ** 2).sum())
    )
    profile["region_hhi"] = region_share

    #Топ-1 регион клиента
    top_region = (
        tx.groupby(["Код клиента", "Регион АЗС"])["Объем"].sum().reset_index().sort_values(["Код клиента", "Объем"], ascending=[True, False]).drop_duplicates("Код клиента").set_index("Код клиента")
    )
    profile["top_region"] = top_region["Регион АЗС"]
    profile["top_region_share"] = (
        top_region["Объем"] / profile["total_volume"]
    ).fillna(0)

    #Время
    tmp = g.agg(
        share_night=("ночь", "mean"),
        share_weekend=("выходной", "mean"),
    )
    profile = profile.join(tmp)

    #Поведение
    profile["unique_suppliers"] = g["Поставщик"].nunique()
    profile["unique_channels"] = g["Канал"].nunique()
    sup_share = (
        tx.groupby(["Код клиента", "Поставщик"])["Объем"].sum()
        .groupby(level=0).apply(lambda s: s.max() / s.sum())
    )
    profile["main_supplier_share"] = sup_share
    profile["price_sensitivity"] = g.apply(_price_sensitivity)

    #Из справочника clients
    c = clients.set_index("Код клиента")
    cols_from_clients = [
        "Статус договора", "Тип клиента", "Офис обслуживания",
        "Заявленный объем в месяц, л", "договор_бессрочный",
        "заявленный_объем_указан", "Дата начала договора",
        "Вид деятельности клиента",
    ]
    profile = profile.join(c[cols_from_clients], how="left")

    #Возраст договора в месяцах относительно последней транзакции
    snapshot = tx["Время транзакции"].max()
    profile["contract_age_months"] = (
        (snapshot - profile["Дата начала договора"]).dt.days / 30
    ).round(1)

    #Выполнение заявленных объемов
    months_active = (profile["activity_days"] / 30).clip(lower=1)
    profile["actual_monthly_volume"] = profile["total_volume"] / months_active
    declared = profile["Заявленный объем в месяц, л"].replace(0, np.nan)
    profile["plan_fulfilment"] = (
        profile["actual_monthly_volume"] / declared
    ).fillna(-1)

    #Сегмент по объёму
    profile["volume_segment"] = pd.qcut(
        profile["total_volume"], q=4,
        labels=["S", "M", "L", "XL"], duplicates="drop"
    )

    return profile.reset_index()

In [60]:
#Профиль поставщика

def build_supplier_profile(tx: pd.DataFrame) -> pd.DataFrame:
    """1 строка на поставщика."""
    g = tx.groupby("Поставщик")

    profile = g.agg(
        tx_count=("Объем", "size"),
        total_volume=("Объем", "sum"),
        unique_clients=("Код клиента", "nunique"),
        unique_stations=("Номер АЗС", "nunique"),
        unique_regions=("Регион АЗС", "nunique"),
        unique_fuel_groups=("Номенклатура", "nunique"),
        own_station_share=("Тип АЗС", "mean"),
        avg_price_ratio=("price_ratio", "mean"),
        median_price_ratio=("price_ratio", "median"),
        std_price_ratio=("price_ratio", "std"),
        unique_channels=("Канал", "nunique"),
        unique_tariff_plans=("Тарифный_план", "nunique"),
    )
    profile["std_price_ratio"] = profile["std_price_ratio"].fillna(0)
    profile["price_stability"] = 1 / (profile["std_price_ratio"] + 1e-3)

    # средняя экономия в рублях на литр
    saving_per_liter = (
        tx.groupby("Поставщик")
        .apply(lambda d: d["saving"].sum() / d["Объем"].sum())
    )
    profile["avg_saving_per_liter"] = saving_per_liter

    # Топливный микс поставщика
    fuel_mix = (
        tx.groupby(["Поставщик", "Номенклатура"])["Объем"].sum()
        .unstack(fill_value=0)
    )
    fuel_mix = fuel_mix.div(
        fuel_mix.sum(axis=1).replace(0, np.nan), axis=0
    ).fillna(0)
    fuel_mix.columns = [f"sup_share_{c}" for c in fuel_mix.columns]
    profile = profile.join(fuel_mix)

    return profile.reset_index()

In [61]:
#Взаимодействия клиента и поставщика

def build_interaction_features(tx: pd.DataFrame) -> pd.DataFrame:
    """
    1 строка на пару (клиент, поставщик) с историей транзакций.
    Это база для:
      - построения CF-матрицы;
      - расчёта target в LTR (composite score);
      - positive-примеры для обучения.
    """
    g = tx.groupby(["Код клиента", "Поставщик"])

    inter = g.agg(
        tx_count=("Объем", "size"),
        volume=("Объем", "sum"),
        spend=("amount", "sum"),
        saving=("saving", "sum"),
        avg_price_ratio=("price_ratio", "mean"),
        std_price_ratio=("price_ratio", "std"),
        first_tx=("Время транзакции", "min"),
        last_tx=("Время транзакции", "max"),
        regions_used=("Регион АЗС", "nunique"),
        stations_used=("Номер АЗС", "nunique"),
        fuel_groups_used=("Номенклатура", "nunique"),
    ).reset_index()

    inter["std_price_ratio"] = inter["std_price_ratio"].fillna(0)
    inter["saving_per_liter"] = inter["saving"] / inter["volume"]
    inter["lifetime_days"] = (inter["last_tx"] - inter["first_tx"]).dt.days

    # доля поставщика в общем объёме клиента
    client_total = inter.groupby("Код клиента")["volume"].transform("sum")
    inter["share_of_client_volume"] = inter["volume"] / client_total

    # доля клиента в общем объёме поставщика
    sup_total = inter.groupby("Поставщик")["volume"].transform("sum")
    inter["share_of_supplier_volume"] = inter["volume"] / sup_total

    # дней с последней транзакции (recency)
    snapshot = tx["Время транзакции"].max()
    inter["days_since_last_tx"] = (snapshot - inter["last_tx"]).dt.days

    return inter

In [62]:
def build_coverage_matrix(tx: pd.DataFrame) -> pd.DataFrame:
    """Долгая таблица (Поставщик, Регион АЗС) с долями АЗС/объёма."""
    cov = (
        tx.groupby(["Поставщик", "Регион АЗС"])
        .agg(stations=("Номер АЗС", "nunique"),
             volume=("Объем", "sum"))
        .reset_index()
    )
    sup_total = cov.groupby("Поставщик")["volume"].transform("sum")
    cov["volume_share_in_supplier"] = cov["volume"] / sup_total
    return cov


def build_fuel_mix_matrix(tx: pd.DataFrame, by: str) -> pd.DataFrame:
    """Wide-матрица (by × fuel_group) с долями объёма."""
    m = (
        tx.groupby([by, "Номенклатура"])["Объем"].sum()
        .unstack(fill_value=0)
    )
    return m.div(m.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)


def build_region_share_client(tx: pd.DataFrame) -> pd.DataFrame:
    """Wide-матрица (Код клиента × Регион АЗС) с долями объёма."""
    m = (
        tx.groupby(["Код клиента", "Регион АЗС"])["Объем"].sum()
        .unstack(fill_value=0)
    )
    return m.div(m.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)


In [63]:
def build_compatibility_features(client_profile: pd.DataFrame,
                                 supplier_profile: pd.DataFrame,
                                 fuel_mix_client: pd.DataFrame,
                                 fuel_mix_supplier: pd.DataFrame,
                                 coverage_matrix: pd.DataFrame,
                                 region_share_client: pd.DataFrame
                                 ) -> pd.DataFrame:
    """
    Декартово произведение клиентов и поставщиков. Для каждой пары:
      - region_coverage_pct  — какая доля объёма клиента попадает в регионы,
        где поставщик работает (важнейший фильтр!);
      - fuel_match_cosine    — косинус между fuel-mix клиента и fuel-mix
        поставщика;
      - fuel_match_overlap   — доля «нужных» (≥5%) топлив клиента, доступных
        у поставщика (≥1%);
      - expected_price_ratio — ожидаемая цена/стела (берём avg по поставщику);
      - estimated_monthly_saving — оценка месячной экономии: сколько бы клиент
        сэкономил, если бы перевёл весь свой объём на этого поставщика.

    Размерность: N_clients × N_suppliers. Для ~4500×5 = 22500 строк, ок.
    """
    clients = client_profile["Код клиента"].unique()
    suppliers = supplier_profile["Поставщик"].unique()

    pairs = pd.MultiIndex.from_product(
        [clients, suppliers], names=["Код клиента", "Поставщик"]
    ).to_frame(index=False)

    # 1. Покрытие регионов клиента
    sup_regions = (
        coverage_matrix.groupby("Поставщик")["Регион АЗС"]
        .apply(set).to_dict()
    )

    # Векторизованное вычисление через словарь
    rsc = region_share_client  # короткое имя

    def _coverage(client_id: str, sup: str) -> float:
        if client_id not in rsc.index:
            return 0.0
        sup_set = sup_regions.get(sup, set())
        cs = rsc.loc[client_id]
        return float(cs[cs.index.isin(sup_set)].sum())

    pairs["region_coverage_pct"] = [
        _coverage(c, s) for c, s in zip(pairs["Код клиента"], pairs["Поставщик"])
    ]

    # 2. Косинус между fuel-mix клиента и поставщика
    cm = fuel_mix_client.copy()
    cm.columns = [str(c) for c in cm.columns]
    sm = fuel_mix_supplier.copy()
    sm.columns = [str(c) for c in sm.columns]
    common = sorted(set(cm.columns) & set(sm.columns))
    cm = cm.reindex(columns=common, fill_value=0)
    sm = sm.reindex(columns=common, fill_value=0)

    cm_arr = cm.reindex(pairs["Код клиента"]).fillna(0).to_numpy()
    sm_arr = sm.reindex(pairs["Поставщик"]).fillna(0).to_numpy()
    dot = (cm_arr * sm_arr).sum(axis=1)
    norm_c = np.linalg.norm(cm_arr, axis=1)
    norm_s = np.linalg.norm(sm_arr, axis=1)
    pairs["fuel_match_cosine"] = np.where(
        (norm_c * norm_s) > 0, dot / (norm_c * norm_s + 1e-9), 0.0
    )

    # 3. Overlap «нужного» и «доступного»
    need_mask = cm_arr >= 0.05
    avail_mask = sm_arr >= 0.01
    pairs["fuel_match_overlap"] = (need_mask & avail_mask).sum(axis=1) / np.maximum(
        need_mask.sum(axis=1), 1
    )

    # 4. Ожидаемое price_ratio = avg_price_ratio поставщика
    sup_pr = supplier_profile.set_index("Поставщик")["avg_price_ratio"]
    pairs["expected_price_ratio"] = pairs["Поставщик"].map(sup_pr)

    # 5. Оценка месячной экономии
    # actual_monthly_volume клиента * (1 - expected_price_ratio) * avg_market_price
    cp = client_profile.set_index("Код клиента")
    pairs["actual_monthly_volume"] = pairs["Код клиента"].map(cp["actual_monthly_volume"])
    # средняя рыночная цена за литр у клиента
    avg_market_price_per_liter = (cp["total_market_spend"] / cp["total_volume"])
    pairs["avg_market_price_per_liter"] = pairs["Код клиента"].map(avg_market_price_per_liter)
    pairs["estimated_monthly_saving"] = (
        pairs["actual_monthly_volume"]
        * pairs["avg_market_price_per_liter"]
        * (1 - pairs["expected_price_ratio"]).clip(lower=0)
    )

    return pairs

In [64]:
def build_activity_clusters(clients: pd.DataFrame,
                            n_clusters: int = 12) -> pd.DataFrame:
    """
    TF-IDF + KMeans по тексту 'Вид деятельности клиента'.
    Кластер пригодится как категориальный признак клиента.
    """
    try:
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.cluster import KMeans
    except ImportError:
        return pd.DataFrame({
            "Код клиента": clients["Код клиента"],
            "activity_cluster": -1,
        })

    text = clients["Вид деятельности клиента"].fillna("не указан").astype(str)
    vec = TfidfVectorizer(max_features=500, ngram_range=(1, 2), min_df=3)
    X = vec.fit_transform(text)
    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    labels = km.fit_predict(X)

    return pd.DataFrame({
        "Код клиента": clients["Код клиента"].values,
        "activity_cluster": labels,
    })

In [65]:
def build_all_features(tx: pd.DataFrame, clients: pd.DataFrame) -> dict:
    """Собирает все фичи. Возвращает dict с DataFrame'ами."""

    print("[1/7] Очистка клиентов…")
    clients = clean_clients(clients)

    print("[2/7] Профиль клиента…")
    client_profile = build_client_profile(tx, clients)

    print("[3/7] Кластеры по виду деятельности…")
    activity = build_activity_clusters(clients)
    client_profile = client_profile.merge(activity, on="Код клиента", how="left")

    print("[4/7] Профиль поставщика…")
    supplier_profile = build_supplier_profile(tx)

    print("[5/7] Взаимодействия клиент×поставщик…")
    interaction = build_interaction_features(tx)

    print("[6/7] Матрицы покрытия и миксов…")
    coverage = build_coverage_matrix(tx)
    fuel_mix_client = build_fuel_mix_matrix(tx, "Код клиента")
    fuel_mix_supplier = build_fuel_mix_matrix(tx, "Поставщик")
    region_share_client = build_region_share_client(tx)

    print("[7/7] Признаки совместимости (декартово произведение)…")
    compatibility = build_compatibility_features(
        client_profile, supplier_profile,
        fuel_mix_client, fuel_mix_supplier,
        coverage, region_share_client,
    )

    return {
        "tx": tx,
        "clients": clients,
        "client_profile": client_profile,
        "supplier_profile": supplier_profile,
        "interaction_features": interaction,
        "coverage_matrix": coverage,
        "fuel_mix_client": fuel_mix_client,
        "fuel_mix_supplier": fuel_mix_supplier,
        "region_share_client": region_share_client,
        "compatibility_features": compatibility,
    }

In [66]:
features = build_all_features(tx, clients)
for name, df in features.items():
  print(f"\n{name}shape={df.shape}")

[1/7] Очистка клиентов…
[2/7] Профиль клиента…


/tmp/ipykernel_3952/1203537489.py:88: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  profile["price_sensitivity"] = g.apply(_price_sensitivity)


[3/7] Кластеры по виду деятельности…
[4/7] Профиль поставщика…


/tmp/ipykernel_3952/446203884.py:27: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda d: d["saving"].sum() / d["Объем"].sum())


[5/7] Взаимодействия клиент×поставщик…
[6/7] Матрицы покрытия и миксов…
[7/7] Признаки совместимости (декартово произведение)…

txshape=(788904, 29)

clientsshape=(4428, 11)

client_profileshape=(4202, 58)

supplier_profileshape=(4, 30)

interaction_featuresshape=(5471, 18)

coverage_matrixshape=(291, 5)

fuel_mix_clientshape=(4202, 15)

fuel_mix_suppliershape=(4, 15)

region_share_clientshape=(4202, 132)

compatibility_featuresshape=(16808, 9)


In [71]:
features['supplier_profile']

,Поставщик,tx_count,total_volume,unique_clients,unique_stations,unique_regions,unique_fuel_groups,own_station_share,avg_price_ratio,median_price_ratio,...,sup_share_АИ-100,sup_share_АИ-92,sup_share_АИ-95,sup_share_АИ-98,sup_share_ГАЗ,sup_share_ДТ,sup_share_КПГ,sup_share_ПРОД. ТОВАРЫ,sup_share_СОПУТСТВУЮЩИЕ ТОВАРЫ,sup_share_ШИНОМОНТАЖ
0,Поставщик1,606546,43750142.09,3642,2823,76,15,0.976188,1.004773,1.000000,...,0.005690,0.127649,0.159622,0.000004,0.000248,0.706535,0.000012,0.000116,0.000017,2.285707e-08
1,Поставщик2,101130,8121606.16,929,1365,78,13,0.806823,1.003688,1.009942,...,0.003357,0.095515,0.131241,0.000018,0.004487,0.765127,0.000000,0.000195,0.000005,0.000000e+00
2,Поставщик3,23733,1641782.48,169,973,94,9,0.935617,1.019427,1.029972,...,0.002485,0.059834,0.194822,0.000000,0.000076,0.742775,0.000000,0.000003,0.000001,0.000000e+00
3,Поставщик4,57495,4610224.23,731,589,43,11,0.992243,0.996845,1.009953,...,0.000000,0.086267,0.120195,0.001124,0.064175,0.728096,0.000000,0.000031,0.000002,0.000000e+00


In [74]:
tables = {
    "tx":                    features["tx"],
    "clients":               features["clients"],
    "client_profile":        features["client_profile"],
    "supplier_profile":      features["supplier_profile"],
    "interaction_features":  features["interaction_features"],
    "coverage_matrix":       features["coverage_matrix"],
    "fuel_mix_client":       features["fuel_mix_client"],
    "fuel_mix_supplier":     features["fuel_mix_supplier"],
    "region_share_client":   features["region_share_client"],
    "compatibility_features":features["compatibility_features"],
}

for name, df in tables.items():
    filepath = f"{name}.csv"
    df.to_csv(filepath, index=False)

print("Все таблицы успешно экспортированы!")

Все таблицы успешно экспортированы!


In [77]:
tx_clean = features['tx']

In [78]:
tx_clean

,Время транзакции,Код клиента,Тип карты,Наименование категории тарифа,Наименование номенклатуры,Объем,Цена стелы,Цена клиента,Тариф клиента,Тип тарифа клиента,...,неделя,час,день_недели,выходной,ночь,Тарифный_план,amount,amount_market,saving,price_ratio
0,2025-01-31 16:17:00,CRRRA668R,Поставщик1.пластик-2,Поставщик1-2,Бензин A95,7.99,62.59,63.84,1.020,Проценты,...,5,16,4,0,0,2,510.0816,500.0941,-9.9875,1.019971
1,2025-01-31 13:32:00,CRRRZAXX8,Поставщик1.пластик,Поставщик1-2,Дизельное топливо,45.00,67.19,68.53,1.020,Проценты,...,5,13,4,0,0,2,3083.8500,3023.5500,-60.3000,1.019943
2,2025-01-31 06:57:00,CRRRA7ZA7,Поставщик1.пластик,Поставщик1-2,Дизельное топливо,10.00,67.19,68.53,1.020,Проценты,...,5,6,4,0,0,2,685.3000,671.9000,-13.4000,1.019943
3,2025-01-31 23:33:00,CRRRA68RA,Поставщик1.пластик,Поставщик1-0,Бензин АИ95 Экто,60.00,61.99,63.23,1.020,Проценты,...,5,23,4,0,1,0,3793.8000,3719.4000,-74.4000,1.020003
4,2025-01-31 18:20:00,CRRRA7Z67,Поставщик1.пластик,Поставщик1-0,Бензин АИ95 Экто,44.53,61.99,64.16,1.035,Проценты,...,5,18,4,0,0,0,2857.0448,2760.4147,-96.6301,1.035006
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
788899,2025-06-30 17:43:00,CRRRZZ7B6,Поставщик3.пластик,Поставщик3-101,Бензин A95,17.63,59.65,59.05,0.990,Проценты,...,27,17,0,0,0,101,1041.0515,1051.6295,10.5780,0.989941
788900,2025-06-30 18:51:00,CRRRZX9RX,Поставщик3.пластик,Поставщик3-101,Дизельное топливо,100.00,70.45,70.45,1.000,Проценты,...,27,18,0,0,0,101,7045.0000,7045.0000,0.0000,1.000000
788901,2025-06-30 20:06:00,CRRRZZ7B6,Поставщик3.пластик,Поставщик3-102,Бензин A95,20.00,57.05,56.48,0.990,Проценты,...,27,20,0,0,0,102,1129.6000,1141.0000,11.4000,0.990009
788902,2025-06-26 22:46:00,CRRRZ6RW6,Поставщик1.пластик,Поставщик1-0,Прод. товары,1.00,159.00,131.97,0.830,Проценты,...,26,22,3,0,1,0,131.9700,159.0000,27.0300,0.830000


In [79]:
"""
Нормализация названий регионов.

Сводит к единому каноничному виду все варианты написания, найденные в
coverage_matrix и в исходных tx. Решает четыре проблемы:

1. Опечатки: "Бресткая" → "Брестская", "Марий ел" → "Марий Эл".
2. Регистр: "Санкт-петербург" → "Санкт-Петербург", "г.Москва" → "Москва".
3. Два формата республик: "Республика Татарстан" и "Татарстан, республика" → один.
4. Раздробление городов федерального значения:
   - Все административные округа Москвы → "Москва"
   - Все районы СПб (Невский, Выборгский, ...) → "Санкт-Петербург"
"""

from __future__ import annotations

import re
import pandas as pd


# ──────────────────────────────────────────────────────────────────────────────
# Словари маппинга
# ──────────────────────────────────────────────────────────────────────────────

# Прямые маппинги: точное совпадение строки → канон
DIRECT_MAP = {
    # опечатки
    "Бресткая область": "Брестская область",
    "Марий ел, республика": "Республика Марий Эл",
    "Марий Эл, республика": "Республика Марий Эл",

    # республики: "X, республика" → "Республика X"
    "Башкортостан, республика": "Республика Башкортостан",
    "Мордовия, республика": "Республика Мордовия",
    "Татарстан, республика": "Республика Татарстан",

    # республики: альтернативные написания → канон
    "Республика Чувашия": "Чувашская Республика",
    "Чувашская республика": "Чувашская Республика",
    "Чеченская республика": "Чеченская Республика",
    "Кабардино-Балкарская Республика": "Республика Кабардино-Балкарская",
    "Карачаево-Черкесская Республика": "Республика Карачаево-Черкесская",
    "Республика Северная Осетия-Алания": "Республика Северная Осетия - Алания",
    "Удмуртская республика": "Удмуртская Республика",
    "Республика Удмуртия": "Удмуртская Республика",
    "Республика Башкортостан": "Республика Башкортостан",
    "Республика Татарстан": "Республика Татарстан",

    # Забайкальский край
    "Забайкальский (край)": "Забайкальский край",

    # ХМАО
    "ХМАО": "Ханты-Мансийский автономный округ - Югра",
    "Ханты-Мансийский автономный округ Югра": "Ханты-Мансийский автономный округ - Югра",

    # ЯНАО
    "ЯНАО": "Ямало-Ненецкий автономный округ",
    "Ямало-Ненецкий АО": "Ямало-Ненецкий автономный округ",

    # Москва: округа и варианты написания → "Москва"
    "г.Москва": "Москва",
    "Восточный административный округ": "Москва",
    "Западный административный округ": "Москва",
    "Северный административный округ": "Москва",
    "Северо-Восточный административный округ": "Москва",
    "Северо-Западный административный округ": "Москва",
    "Юго-Восточный административный округ": "Москва",
    "Юго-Западный административный округ": "Москва",
    "Южный административный округ": "Москва",
    "Центральный административный округ": "Москва",
    "Новомосковский административный округ": "Москва",
    "Троицкий административный округ": "Москва",

    # СПб: районы и варианты написания → "Санкт-Петербург"
    "г.Санкт-Петербург": "Санкт-Петербург",
    "Санкт-петербург": "Санкт-Петербург",
    "Выборгский район": "Санкт-Петербург",
    "Калининский район": "Санкт-Петербург",
    "Кировский район": "Санкт-Петербург",
    "Колпинский район": "Санкт-Петербург",
    "Красногвардейский район": "Санкт-Петербург",
    "Красносельский район": "Санкт-Петербург",
    "Курортный район": "Санкт-Петербург",
    "Московский район": "Санкт-Петербург",
    "Невский район": "Санкт-Петербург",
    "Приморский район": "Санкт-Петербург",
    "Фрунзенский район": "Санкт-Петербург",
    "Центральный район": "Санкт-Петербург",
}


# Регионы за пределами РФ — помечаем страной (полезно для аналитики)
FOREIGN_REGIONS = {
    # Беларусь
    "Брестская область": "BY",
    "Витебская область": "BY",
    "Гомельская область": "BY",
    "Гродненская область": "BY",
    "Минская область": "BY",
    "Могилевская область": "BY",
    # Казахстан
    "Акмолинская область": "KZ",
    "Северо-Казахстанская область": "KZ",
    # Абхазия
    "Гагрский район": "AB",
}


# ──────────────────────────────────────────────────────────────────────────────
# Основная функция
# ──────────────────────────────────────────────────────────────────────────────

def normalize_region(name: str) -> str:
    """Возвращает канонизированное название региона."""
    if pd.isna(name):
        return name

    s = str(name).strip()

    # 1. Прямой маппинг
    if s in DIRECT_MAP:
        return DIRECT_MAP[s]

    # 2. Унификация регистра первой буквы для случая "Санкт-петербург"-подобных
    #    (на случай если найдутся новые варианты)
    s_norm = re.sub(r"\bпетербург\b", "Петербург", s, flags=re.IGNORECASE)
    s_norm = re.sub(r"^г\.", "", s_norm).strip()
    if s_norm in DIRECT_MAP:
        return DIRECT_MAP[s_norm]

    return s


def get_country_code(canonical_region: str) -> str:
    """Двухбуквенный код страны для каноничного региона. По умолчанию RU."""
    return FOREIGN_REGIONS.get(canonical_region, "RU")


def normalize_regions_in_df(df: pd.DataFrame,
                            col: str = "Регион АЗС") -> pd.DataFrame:
    """
    Применяет normalize_region к указанной колонке.
    Возвращает копию датафрейма с дополнительной колонкой 'Страна'.
    """
    df = df.copy()
    df[col] = df[col].map(normalize_region)
    df["Страна"] = df[col].map(get_country_code)
    return df


def report_normalization(before: pd.Series, after: pd.Series) -> dict:
    """Краткая статистика: что было / что стало."""
    return {
        "regions_before": before.nunique(),
        "regions_after": after.nunique(),
        "merged_count": before.nunique() - after.nunique(),
    }


if __name__ == "__main__":
    cov = pd.read_csv("coverage_matrix.csv")
    cov_n = normalize_regions_in_df(cov)

    print("Регионов до:", cov["Регион АЗС"].nunique())
    print("Регионов после:", cov_n["Регион АЗС"].nunique())
    print()
    print("По странам:")
    print(cov_n["Страна"].value_counts())

Регионов до: 132
Регионов после: 88

По странам:
Страна
RU    277
BY     11
KZ      2
AB      1
Name: count, dtype: int64


In [ ]:
# Финальный автономный блок: нормализация регионов, target definition и русификация признаков
# Этот блок не использует внешние .py-файлы и может выполняться после запуска предыдущих ячеек ноутбука.

from dataclasses import dataclass


@dataclass
class ScoreConfig:
    """Настройки расчета интегральной полезности поставщика для клиента."""

    w_economy: float = 0.50
    w_coverage: float = 0.25
    w_fuel: float = 0.15
    w_stability: float = 0.10
    min_region_coverage: float = 0.30
    min_fuel_overlap: float = 0.50
    saving_clip_pct: float = 0.10
    saving_floor_pct: float = -0.05


def _normalize_economy(expected_price_ratio: pd.Series, cfg: ScoreConfig) -> pd.Series:
    """Переводит ожидаемое отношение цены к стеле в score от 0 до 1."""
    benefit = 1 - expected_price_ratio
    benefit = benefit.clip(lower=cfg.saving_floor_pct, upper=cfg.saving_clip_pct)
    return (benefit - cfg.saving_floor_pct) / (cfg.saving_clip_pct - cfg.saving_floor_pct)


def _normalize_stability(supplier_profile: pd.DataFrame) -> pd.Series:
    """Стабильность цены поставщика: обратная величина разброса price_ratio."""
    stab = supplier_profile.set_index("Поставщик")["price_stability"]
    log_stab = np.log1p(stab)
    norm = (log_stab - log_stab.min()) / (log_stab.max() - log_stab.min() + 1e-9)
    return norm.rename("stability_score")


def build_target(
    compatibility_features: pd.DataFrame,
    supplier_profile: pd.DataFrame,
    interaction_features: pd.DataFrame,
    client_profile: pd.DataFrame,
    cfg: ScoreConfig | None = None,
) -> pd.DataFrame:
    """Считает composite_score для всех пар клиент-поставщик."""
    if cfg is None:
        cfg = ScoreConfig()

    df = compatibility_features.copy()
    df["score_economy"] = _normalize_economy(df["expected_price_ratio"], cfg)
    df["score_coverage"] = df["region_coverage_pct"].clip(0, 1)
    df["score_fuel"] = df["fuel_match_overlap"].clip(0, 1)
    df["score_stability"] = df["Поставщик"].map(_normalize_stability(supplier_profile)).fillna(0)

    df["composite_score_raw"] = (
        cfg.w_economy * df["score_economy"]
        + cfg.w_coverage * df["score_coverage"]
        + cfg.w_fuel * df["score_fuel"]
        + cfg.w_stability * df["score_stability"]
    )

    df["hard_filter_pass"] = (
        (df["region_coverage_pct"] >= cfg.min_region_coverage)
        & (df["fuel_match_overlap"] >= cfg.min_fuel_overlap)
    )
    df["composite_score"] = np.where(df["hard_filter_pass"], df["composite_score_raw"], 0.0)

    history_pairs = set(zip(interaction_features["Код клиента"], interaction_features["Поставщик"]))
    df["has_history"] = [
        (client_id, supplier) in history_pairs
        for client_id, supplier in zip(df["Код клиента"], df["Поставщик"])
    ]

    component_cols = ["score_economy", "score_coverage", "score_fuel", "score_stability"]
    weighted = df[component_cols].mul(
        [cfg.w_economy, cfg.w_coverage, cfg.w_fuel, cfg.w_stability], axis=1
    )
    weighted.columns = ["экономия", "покрытие_регионов", "топливный_микс", "стабильность_цен"]
    df["top_factor"] = weighted.idxmax(axis=1)
    df["top_factor"] = np.where(df["hard_filter_pass"], df["top_factor"], "не_проходит_фильтры")

    df["score_pct"] = (df["composite_score"] * 100).round(1)
    df["expected_savings_pct"] = ((1 - df["expected_price_ratio"]) * 100).round(2)
    return df.drop(columns=["composite_score_raw"])


def get_top_n_recommendations(target_df: pd.DataFrame, n: int = 3, only_passing: bool = True) -> pd.DataFrame:
    """Возвращает top-N поставщиков для каждого клиента."""
    df = target_df.copy()
    if only_passing:
        df = df[df["hard_filter_pass"]]
    df = df.sort_values(["Код клиента", "composite_score"], ascending=[True, False])
    df["rank"] = df.groupby("Код клиента").cumcount() + 1
    return df[df["rank"] <= n].reset_index(drop=True)


def detect_multi_supplier_cases(target_df: pd.DataFrame, cfg: ScoreConfig | None = None) -> pd.DataFrame:
    """Находит клиентов, которым может потребоваться связка из нескольких поставщиков."""
    if cfg is None:
        cfg = ScoreConfig()
    max_cov = target_df.groupby("Код клиента")["region_coverage_pct"].max()
    return pd.DataFrame({
        "Код клиента": max_cov.index,
        "max_single_supplier_coverage": max_cov.values,
        "needs_multi_supplier": (max_cov < 0.7).values,
    })


def diagnose_target(target_df: pd.DataFrame) -> dict:
    """Сводная диагностика распределения composite_score."""
    return {
        "total_pairs": len(target_df),
        "passing_hard_filters": int(target_df["hard_filter_pass"].sum()),
        "with_history": int(target_df["has_history"].sum()),
        "pairs_with_score_above_0_7": int((target_df["composite_score"] > 0.7).sum()),
        "pairs_with_score_above_0_5": int((target_df["composite_score"] > 0.5).sum()),
        "score_mean": float(target_df["composite_score"].mean()),
        "score_median": float(target_df["composite_score"].median()),
        "score_distribution": target_df["composite_score"].describe().to_dict(),
        "top_factor_distribution": target_df["top_factor"].value_counts().to_dict(),
    }


FEATURE_NAME_RU = {
    "tx": "Транзакции",
    "clients": "Клиенты",
    "client_profile": "Профиль клиента",
    "supplier_profile": "Профиль поставщика",
    "interaction_features": "Взаимодействия клиент-поставщик",
    "coverage_matrix": "Матрица покрытия регионов",
    "fuel_mix_client": "Топливный микс клиента",
    "fuel_mix_supplier": "Топливный микс поставщика",
    "region_share_client": "Доли регионов клиента",
    "compatibility_features": "Признаки совместимости",
    "target": "Целевая таблица",
    "recommendations_top3": "Топ-3 рекомендации",
    "multi_supplier_cases": "Кейсы нескольких поставщиков",
    "amount": "Сумма клиента",
    "amount_market": "Сумма по цене стелы",
    "saving": "Экономия",
    "price_ratio": "Отношение цены клиента к цене стелы",
    "tx_count": "Количество транзакций",
    "total_volume": "Общий объем",
    "avg_volume_per_tx": "Средний объем транзакции",
    "median_volume": "Медианный объем транзакции",
    "std_volume": "Стандартное отклонение объема",
    "first_tx": "Первая транзакция",
    "last_tx": "Последняя транзакция",
    "activity_days": "Дней активности",
    "tx_per_day": "Транзакций в день",
    "total_spend": "Общие траты",
    "total_market_spend": "Общие траты по цене стелы",
    "total_saving": "Общая экономия",
    "avg_ticket": "Средний чек",
    "avg_price_ratio": "Среднее отношение цены к стеле",
    "saving_pct": "Доля экономии",
    "fuel_entropy": "Энтропия топливного микса",
    "fuel_diversity": "Разнообразие топлива",
    "unique_regions": "Количество регионов",
    "unique_stations": "Количество АЗС",
    "own_station_share": "Доля собственных АЗС",
    "region_hhi": "Концентрация регионов HHI",
    "top_region": "Основной регион",
    "top_region_share": "Доля основного региона",
    "share_night": "Доля ночных транзакций",
    "share_weekend": "Доля транзакций в выходные",
    "unique_suppliers": "Количество поставщиков",
    "unique_channels": "Количество каналов",
    "main_supplier_share": "Доля основного поставщика",
    "price_sensitivity": "Ценовая чувствительность",
    "contract_age_months": "Возраст договора, месяцев",
    "actual_monthly_volume": "Фактический месячный объем",
    "plan_fulfilment": "Выполнение заявленного объема",
    "volume_segment": "Сегмент по объему",
    "activity_cluster": "Кластер вида деятельности",
    "unique_clients": "Количество клиентов",
    "unique_fuel_groups": "Количество топливных групп",
    "median_price_ratio": "Медианное отношение цены к стеле",
    "std_price_ratio": "Стандартное отклонение цены к стеле",
    "price_stability": "Стабильность цены",
    "avg_saving_per_liter": "Средняя экономия на литр",
    "unique_tariff_plans": "Количество тарифных планов",
    "volume": "Объем",
    "spend": "Траты",
    "regions_used": "Использовано регионов",
    "stations_used": "Использовано АЗС",
    "fuel_groups_used": "Использовано топливных групп",
    "saving_per_liter": "Экономия на литр",
    "lifetime_days": "Дней с первой до последней транзакции",
    "share_of_client_volume": "Доля объема клиента",
    "share_of_supplier_volume": "Доля объема поставщика",
    "days_since_last_tx": "Дней с последней транзакции",
    "stations": "Количество АЗС",
    "volume_share_in_supplier": "Доля объема у поставщика",
    "region_coverage_pct": "Покрытие регионов клиента",
    "fuel_match_cosine": "Сходство топливного микса",
    "fuel_match_overlap": "Пересечение топливного микса",
    "expected_price_ratio": "Ожидаемое отношение цены к стеле",
    "avg_market_price_per_liter": "Средняя цена стелы за литр",
    "estimated_monthly_saving": "Оценка месячной экономии",
    "score_economy": "Скор экономии",
    "score_coverage": "Скор покрытия",
    "score_fuel": "Скор топливного микса",
    "score_stability": "Скор стабильности цены",
    "hard_filter_pass": "Прошел жесткие фильтры",
    "composite_score": "Композитный скор",
    "has_history": "Есть история взаимодействий",
    "top_factor": "Главный фактор",
    "score_pct": "Композитный скор, %",
    "expected_savings_pct": "Ожидаемая экономия, %",
    "max_single_supplier_coverage": "Максимальное покрытие одним поставщиком",
    "needs_multi_supplier": "Нужны несколько поставщиков",
    "rank": "Ранг",
}


def russian_feature_name(col: str) -> str:
    """Переводит техническое имя признака в русскоязычное название."""
    if col in FEATURE_NAME_RU:
        return FEATURE_NAME_RU[col]
    if col.startswith("share_"):
        return "Доля топлива " + col.removeprefix("share_")
    if col.startswith("sup_share_"):
        return "Доля топлива у поставщика " + col.removeprefix("sup_share_")
    if col.startswith("sup_"):
        return "Поставщик: " + russian_feature_name(col.removeprefix("sup_"))
    return col


def to_russian_feature_names(df: pd.DataFrame) -> pd.DataFrame:
    """Возвращает копию датафрейма с русскими названиями признаков."""
    return df.rename(columns={c: russian_feature_name(str(c)) for c in df.columns})


# Нормализуем регионы до финального построения признаков и таргета.
tx_normalized = normalize_regions_in_df(tx, "Регион АЗС")
normalization_report = report_normalization(tx["Регион АЗС"], tx_normalized["Регион АЗС"])
print("Нормализация регионов:", normalization_report)

features = build_all_features(tx_normalized, clients)
target = build_target(
    compatibility_features=features["compatibility_features"],
    supplier_profile=features["supplier_profile"],
    interaction_features=features["interaction_features"],
    client_profile=features["client_profile"],
)
recommendations_top3 = get_top_n_recommendations(target, n=3)
multi_supplier_cases = detect_multi_supplier_cases(target)
target_diagnostics = diagnose_target(target)

print("Диагностика target:")
for key, value in target_diagnostics.items():
    if key not in {"score_distribution", "top_factor_distribution"}:
        print(f"  {key}: {value}")

# Русскоязычные версии таблиц для анализа, отчета и экспорта.
features_ru = {name: to_russian_feature_names(df) for name, df in features.items()}
target_ru = to_russian_feature_names(target)
recommendations_top3_ru = to_russian_feature_names(recommendations_top3)
multi_supplier_cases_ru = to_russian_feature_names(multi_supplier_cases)

russian_tables = {
    **features_ru,
    "target": target_ru,
    "recommendations_top3": recommendations_top3_ru,
    "multi_supplier_cases": multi_supplier_cases_ru,
}

OUTPUT_DIR.mkdir(exist_ok=True)
for name, df in russian_tables.items():
    df.to_csv(OUTPUT_DIR / f"{name}_ru.csv", index=False)

print(f"Экспортировано русскоязычных таблиц: {len(russian_tables)}")
display(recommendations_top3_ru.head(10))